# Databricks notebook source
E-Commerce Data Pipeline on Databricks Serverless
This notebook builds a complete e-commerce data pipeline using the Medallion Architecture.
The flow is:
`CSV files in a Unity Catalog Volume -> Landing -> Bronze -> Silver -> Gold -> Reconciliation`


The notebook is deliberately written step by step. Run the cells from top to bottom the first time.
## Lets understand what we are building
The source files contain customers, orders, order items, and inventory. They intentionally include duplicate and invalid records. This notebook keeps the raw data, cleans it in a separate layer, and produces business-ready Delta tables.

### Why is this used?
Separating raw, cleaned, and reporting data makes the pipeline easier to trust. If a report looks wrong, we can trace the result back through each layer without changing the original file.

## Lets create the project configuration
Change only `CATALOG_NAME` if your workspace catalog has a different name. You can find the catalog name in the Catalog panel on the left side of Databricks.

In [0]:
from datetime import datetime
import re

from delta.tables import DeltaTable
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType
from pyspark.sql.window import Window


CATALOG_NAME = "testad"
PROJECT_SCHEMA = "ecommerce_project"
VOLUME_NAME = "landing_files"

LANDING_SCHEMA = "ecommerce_landing"
BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{PROJECT_SCHEMA}/{VOLUME_NAME}"
RUN_TIMESTAMP = datetime.utcnow()

EXPECTED_FILES = {
    "orders": "orders.csv",
    "order_items": "order_items.csv",
    "customers": "customers.csv",
    "inventory": "inventory.csv",
}


def sql_identifier(name: str) -> str:
    """Return a safely quoted Unity Catalog identifier."""
    if not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", name):
        raise ValueError(
            f"'{name}' is not a valid catalog, schema, or table name. "
            "Use letters, numbers, and underscores only, starting with a letter."
        )
    return f"`{name}`"


def table_name(schema: str, table: str) -> str:
    return ".".join([sql_identifier(CATALOG_NAME), sql_identifier(schema), sql_identifier(table)])


def show_count(label: str, dataframe: DataFrame) -> int:
    """Print a consistent row-count check after each important stage."""
    row_count = dataframe.count()
    print(f"{label}: {row_count:,} rows")
    return row_count


def latest_by_key(dataframe: DataFrame, primary_key: str) -> DataFrame:
    """Keep the latest business key using source time and a stable same-file tie-breaker."""
    stable_hash = F.xxhash64(*[F.col(column_name) for column_name in sorted(dataframe.columns)])
    latest_window = Window.partitionBy(primary_key).orderBy(
        F.col("bronze_ingestion_timestamp").desc_nulls_last(),
        F.to_timestamp("load_ts").desc_nulls_last(),
        F.col("source_file_name").desc_nulls_last(),
        stable_hash.desc(),
    )
    return dataframe.withColumn("_row_number", F.row_number().over(latest_window)).where(F.col("_row_number") == 1).drop("_row_number")


def overwrite_delta(dataframe: DataFrame, destination: str, partition_columns: list[str] | None = None) -> None:
    """Write a deterministic Delta table that is safe to rebuild from its upstream layer."""
    writer = dataframe.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if partition_columns:
        writer = writer.partitionBy(*partition_columns)
    writer.saveAsTable(destination)


for required_name in [CATALOG_NAME, PROJECT_SCHEMA, VOLUME_NAME, LANDING_SCHEMA, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    sql_identifier(required_name)

print(f"Selected catalog: {CATALOG_NAME}")
print(f"Landing Volume: {VOLUME_PATH}")


Selected catalog: testad
Landing Volume: /Volumes/testad/ecommerce_project/landing_files


/home/spark-52061774-ea8b-47b9-8136-b2/.ipykernel/84/command-4835727616214257-936994672:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_TIMESTAMP = datetime.utcnow()


## Lets create the Unity Catalog schemas and Volume
### Why is this used?
A Unity Catalog Volume is the serverless-safe location for the input CSV files. Tables will be created in separate schemas so each Medallion layer is easy to find in Catalog Explorer.

In [0]:
for schema_name in [PROJECT_SCHEMA, LANDING_SCHEMA, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {sql_identifier(CATALOG_NAME)}.{sql_identifier(schema_name)}")

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {sql_identifier(CATALOG_NAME)}.{sql_identifier(PROJECT_SCHEMA)}.{sql_identifier(VOLUME_NAME)}"
)

print("Schemas and Volume are ready.")
print(f"Upload the four CSV files to: {VOLUME_PATH}")

Schemas and Volume are ready.
Upload the four CSV files to: /Volumes/testad/ecommerce_project/landing_files


## Lets upload the source files before running the pipeline
1. In the left sidebar, open **Catalog**.
2. Open your catalog, then `ecommerce_project`, then `landing_files`.
3. Select **Upload files** and upload `orders.csv`, `order_items.csv`, `customers.csv`, and `inventory.csv` from `Shared/Data`.
4. Return here and run the next cell.
Keep the uploaded files at the root of the Volume. The notebook looks for the four exact file names.

In [0]:
# Auto-copy source CSVs from workspace into the Volume if not already present
SOURCE_DIR = "file:/Workspace/Shared/Data"
for file_name in EXPECTED_FILES.values():
    dest = f"{VOLUME_PATH}/{file_name}"
    try:
        dbutils.fs.ls(dest)
    except Exception:
        dbutils.fs.cp(f"{SOURCE_DIR}/{file_name}", dest)

available_files = {item.name.rstrip("/") for item in dbutils.fs.ls(VOLUME_PATH)}
missing_files = sorted(set(EXPECTED_FILES.values()) - available_files)

if missing_files:
    raise FileNotFoundError(
        "The required CSV files are not all in the Unity Catalog Volume. "
        f"Missing: {', '.join(missing_files)}. Upload them to {VOLUME_PATH} and run this cell again."
    )

print("All required input files are available:")
for file_name in sorted(EXPECTED_FILES.values()):
    print(f"- {file_name}")

All required input files are available:
- customers.csv
- inventory.csv
- order_items.csv
- orders.csv


## Lets understand the source data

### Why is this used?

CSV files do not enforce business data types. We first read every value as text so that formatting problems do not silently change the meaning of a value before validation.



In [0]:
source_dataframes: dict[str, DataFrame] = {}

for dataset_name, file_name in EXPECTED_FILES.items():
    file_path = f"{VOLUME_PATH}/{file_name}"
    dataframe = (
        spark.read.option("header", True)
        .option("inferSchema", False)
        .option("mode", "FAILFAST")
        .csv(file_path)
    )
    source_dataframes[dataset_name] = dataframe
    print(f"\n{dataset_name}: {file_path}")
    dataframe.printSchema()
    show_count(f"{dataset_name} source", dataframe)
    display(dataframe.limit(5))


orders: /Volumes/testad/ecommerce_project/landing_files/orders.csv
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- discount_amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- load_ts: string (nullable = true)

orders source: 50,150 rows


order_id,customer_id,order_date,status,total_amount,discount_amount,payment_method,warehouse_id,region,load_ts
ORD00000001,null,2025-03-09 21:35:49,placed,18079.31,4993.93,debit_card,WH-CHE,North,2025-04-17 06:00:00
ORD00000002,CUST005368,2025-01-11 07:38:06,delivered,18686.12,5117.37,UPI,WH-BLR,South,2025-04-17 06:00:00
ORD00000003,CUST008905,2025-03-06 15:59:27,unknown,37342.13,5086.63,credit_card,WH-CHE,North,2025-04-17 06:00:00
ORD00000004,CUST007879,2025-02-23 03:43:08,delivered,48334.85,8979.02,credit_card,WH-BLR,Central,2025-04-17 06:00:00
ORD00000005,CUST008514,2025-01-23 03:58:49,PENDING,5811.5,1665.29,UPI,WH-HYD,West,2025-04-17 06:00:00



order_items: /Volumes/testad/ecommerce_project/landing_files/order_items.csv
root
 |-- item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- sku_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- line_total: string (nullable = true)
 |-- load_ts: string (nullable = true)

order_items source: 200,000 rows


item_id,order_id,sku_id,product_name,category,quantity,unit_price,line_total,load_ts
ITEM000000001,ORD00022044,SKU00069,Organized grid-enabled access,Beauty,null,-882.67,null,2025-04-17 06:00:00
ITEM000000002,ORD00029814,SKU01988,Organized holistic methodology,Sports,4,10382.79,41531.16,2025-04-17 06:00:00
ITEM000000003,ORD00005483,SKU02931,Business-focused attitude-oriented matrix,Grocery,3,949.86,2849.58,2025-04-17 06:00:00
ITEM000000004,ORD00021103,SKU02099,Reactive analyzing time-frame,Electronics,10,11822.57,118225.7,2025-04-17 06:00:00
ITEM000000005,ORD00009947,SKU02318,Enterprise-wide background hub,Clothing,6,2262.3,13573.8,2025-04-17 06:00:00



customers: /Volumes/testad/ecommerce_project/landing_files/customers.csv
root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- load_ts: string (nullable = true)

customers source: 10,000 rows


customer_id,first_name,last_name,email,phone,city,state,region,signup_date,is_active,load_ts
CUST000001,Liam,Chaudry,Udant@,8196001338,Kishanganj,Gujarat,North,2022-02-08 20:28:06,1,2025-04-17 06:00:00
CUST000002,Arunima,Ahuja,ckannan@example.net,+916542351161,Hapur,Punjab,South,2022-12-13 17:53:58,1,2025-04-17 06:00:00
CUST000003,Kritika,Brar,caleb78@example.org,4959310341,Salem,Madhya Pradesh,North,2024-11-17 05:11:07,1,2025-04-17 06:00:00
CUST000004,Isha,Kadakia,sudiksha52@example.com,4192832764,Anantapur,Uttarakhand,Central,2023-10-18 10:23:08,1,2025-04-17 06:00:00
CUST000005,Nandini,Loyal,karnikazad@example.com,+913953767242,Kishanganj,Nagaland,North,2022-05-26 13:12:42,1,2025-04-17 06:00:00



inventory: /Volumes/testad/ecommerce_project/landing_files/inventory.csv
root
 |-- sku_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- stock_quantity: string (nullable = true)
 |-- reorder_level: string (nullable = true)
 |-- unit_cost: string (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- load_ts: string (nullable = true)

inventory source: 5,000 rows


sku_id,product_name,category,warehouse_id,stock_quantity,reorder_level,unit_cost,last_updated,load_ts
SKU00001,Open-architected maximized time-frame,Books,WH-MUM,null,51,9076.16,2025-04-09 07:14:39,2025-04-17 06:00:00
SKU00002,Ergonomic empowering workforce,Clothing,WH-DEL,557,86,6597.28,2025-04-05 11:20:13,2025-04-17 06:00:00
SKU00003,Multi-lateral dynamic utilization,Beauty,WH-DEL,1798,90,8386.89,2025-04-07 04:00:04,2025-04-17 06:00:00
SKU00004,Integrated 6thgeneration frame,Grocery,WH-MUM,1839,23,13021.31,2025-04-05 10:08:15,2025-04-17 06:00:00
SKU00005,Upgradable mission-critical implementation,Books,WH-DEL,76,99,6116.95,2025-04-14 03:44:16,2025-04-17 06:00:00


## Lets create the Landing layer

Landing is the auditable raw-file history. Every source column remains a string, and two metadata columns show when and from which file the data arrived.



In [0]:
landing_tables: dict[str, DataFrame] = {}

for dataset_name, dataframe in source_dataframes.items():
    landing_dataframe = (
        dataframe.select(
            *[F.col(column_name).cast("string").alias(column_name) for column_name in dataframe.columns],
            F.col("_metadata.file_path").cast("string").alias("source_file_name")
        )
        .withColumn("landing_timestamp", F.lit(RUN_TIMESTAMP).cast("timestamp"))
    )
    destination = table_name(LANDING_SCHEMA, dataset_name)
    landing_dataframe.write.format("delta").mode("append").saveAsTable(destination)
    landing_tables[dataset_name] = spark.table(destination)
    show_count(f"Landing {dataset_name}", landing_tables[dataset_name])

print("Landing layer complete. Landing keeps every file-load history, including repeated uploads.")

Landing orders: 50,150 rows
Landing order_items: 200,000 rows
Landing customers: 10,000 rows
Landing inventory: 5,000 rows
Landing layer complete. Landing keeps every file-load history, including repeated uploads.


## Lets create the Bronze layer
### Why is this used?
Bronze preserves the raw source values while adding ingestion metadata and a date partition. This gives us a stable source for cleaning while keeping the original fields untouched.


In [0]:
bronze_tables: dict[str, DataFrame] = {}

for dataset_name in EXPECTED_FILES:
    bronze_dataframe = (
        spark.table(table_name(LANDING_SCHEMA, dataset_name))
        .withColumn("bronze_ingestion_timestamp", F.col("landing_timestamp"))
        .withColumn("load_date", F.to_date("landing_timestamp"))
    )
    destination = table_name(BRONZE_SCHEMA, dataset_name)
    overwrite_delta(bronze_dataframe, destination, ["load_date"])
    bronze_tables[dataset_name] = spark.table(destination)
    show_count(f"Bronze {dataset_name}", bronze_tables[dataset_name])

print("Bronze layer complete.")

Bronze orders: 50,150 rows
Bronze order_items: 200,000 rows
Bronze customers: 10,000 rows
Bronze inventory: 5,000 rows
Bronze layer complete.


## Lets create the Silver layer for orders
### Why is this used?
Silver is where we remove duplicate re-exports, apply business data types, and separate invalid records. Invalid orders are retained in a quarantine table instead of disappearing.

In [0]:
orders_latest = latest_by_key(bronze_tables["orders"], "order_id")

if orders_latest.where(F.col("order_id").isNull() | (F.trim("order_id") == "")).limit(1).count() > 0:
    raise ValueError(
        "Orders contains a missing order_id. order_id is the primary key, so the pipeline stopped before producing unreliable Silver data."
    )

orders_typed = orders_latest.select(
    F.trim("order_id").alias("order_id"),
    F.trim("customer_id").alias("customer_id"),
    F.to_timestamp("order_date").alias("order_date"),
    F.lower(F.trim("status")).alias("status"),
    F.col("total_amount").cast(DoubleType()).alias("total_amount"),
    F.col("discount_amount").cast(DoubleType()).alias("discount_amount"),
    F.trim("payment_method").alias("payment_method"),
    F.trim("warehouse_id").alias("warehouse_id"),
    F.trim("region").alias("region"),
    F.to_timestamp("load_ts").alias("source_load_timestamp"),
    F.col("bronze_ingestion_timestamp"),
    F.col("load_date"),
)

valid_statuses = ["placed", "shipped", "delivered", "cancelled"]
orders_with_reason = orders_typed.withColumn(
    "quarantine_reason",
    F.concat_ws(
        "; ",
        F.when(~F.col("status").isin(valid_statuses) | F.col("status").isNull(), F.lit("Invalid status")),
        F.when(F.col("order_date").isNull(), F.lit("Invalid order date")),
        F.when(F.col("total_amount").isNull() | (F.col("total_amount") <= 0), F.lit("Total amount must be positive")),
        F.when(F.col("customer_id").isNull() | (F.trim("customer_id") == ""), F.lit("Missing customer_id")),
    ),
)

invalid_orders = orders_with_reason.where(F.col("quarantine_reason") != "")
valid_orders = orders_with_reason.where(F.col("quarantine_reason") == "").drop("quarantine_reason")

overwrite_delta(valid_orders, table_name(SILVER_SCHEMA, "orders"))
overwrite_delta(invalid_orders, table_name(SILVER_SCHEMA, "orders_quarantine"))

show_count("Silver orders", valid_orders)
show_count("Orders quarantine", invalid_orders)

Silver orders: 40,591 rows
Orders quarantine: 9,409 rows


9409

## Lets create the Silver layer for order items
Order items are checked independently because quantity and unit price determine the value of each line item.

In [0]:
order_items_latest = latest_by_key(bronze_tables["order_items"], "item_id")
order_items_typed = order_items_latest.select(
    F.trim("item_id").alias("item_id"),
    F.trim("order_id").alias("order_id"),
    F.trim("sku_id").alias("sku_id"),
    F.trim("product_name").alias("product_name"),
    F.trim("category").alias("category"),
    F.col("quantity").cast(IntegerType()).alias("quantity"),
    F.col("unit_price").cast(DoubleType()).alias("unit_price"),
    # Line total is derived from quantity multiplied by unit price to avoid relying on a malformed source total.
    F.round(F.col("quantity").cast(DoubleType()) * F.col("unit_price").cast(DoubleType()), 2).alias("line_total"),
    F.to_timestamp("load_ts").alias("source_load_timestamp"),
    F.col("bronze_ingestion_timestamp"),
    F.col("load_date"),
)

items_with_reason = order_items_typed.withColumn(
    "quarantine_reason",
    F.concat_ws(
        "; ",
        F.when(F.col("quantity").isNull() | (F.col("quantity") <= 0), F.lit("Quantity must be positive")),
        F.when(F.col("unit_price").isNull() | (F.col("unit_price") <= 0), F.lit("Unit price must be positive")),
    ),
)

invalid_order_items = items_with_reason.where(F.col("quarantine_reason") != "")
valid_order_items = items_with_reason.where(F.col("quarantine_reason") == "").drop("quarantine_reason")

overwrite_delta(valid_order_items, table_name(SILVER_SCHEMA, "order_items"))
overwrite_delta(invalid_order_items, table_name(SILVER_SCHEMA, "order_items_quarantine"))

show_count("Silver order items", valid_order_items)
show_count("Order items quarantine", invalid_order_items)

Silver order items: 195,044 rows
Order items quarantine: 4,956 rows


4956

## Lets create the Silver layer for inventory and customers
### Why is this used?
Inventory with no stock quantity cannot support a stock calculation, so it is excluded from the clean table. Customer email issues are audited but retained because the customer record can still be useful for sales analysis.

In [0]:
inventory_latest = latest_by_key(bronze_tables["inventory"], "sku_id")
inventory_typed = inventory_latest.select(
    F.trim("sku_id").alias("sku_id"),
    F.trim("product_name").alias("product_name"),
    F.trim("category").alias("category"),
    F.trim("warehouse_id").alias("warehouse_id"),
    F.col("stock_quantity").cast(IntegerType()).alias("stock_quantity"),
    F.col("reorder_level").cast(IntegerType()).alias("reorder_level"),
    F.col("unit_cost").cast(DoubleType()).alias("unit_cost"),
    F.to_timestamp("last_updated").alias("last_updated"),
    F.to_timestamp("load_ts").alias("source_load_timestamp"),
    F.col("bronze_ingestion_timestamp"),
    F.col("load_date"),
)
valid_inventory = (
    inventory_typed.where(F.col("stock_quantity").isNotNull())
    .withColumn("has_negative_stock", F.col("stock_quantity") < 0)
)
overwrite_delta(valid_inventory, table_name(SILVER_SCHEMA, "inventory"))
show_count("Silver inventory", valid_inventory)
print(f"Inventory records excluded because stock quantity is null: {inventory_typed.where(F.col('stock_quantity').isNull()).count():,}")
print(f"Inventory records retained with negative stock warning: {valid_inventory.where(F.col('has_negative_stock')).count():,}")

customers_latest = latest_by_key(bronze_tables["customers"], "customer_id")
customers_typed = customers_latest.select(
    F.trim("customer_id").alias("customer_id"),
    F.trim("first_name").alias("first_name"),
    F.trim("last_name").alias("last_name"),
    F.trim("email").alias("email"),
    F.trim("phone").alias("phone"),
    F.trim("city").alias("city"),
    F.trim("state").alias("state"),
    F.trim("region").alias("region"),
    F.to_date("signup_date").alias("signup_date"),
    F.col("is_active").cast("boolean").alias("is_active"),
    F.to_timestamp("load_ts").alias("source_load_timestamp"),
    F.col("bronze_ingestion_timestamp"),
    F.col("load_date"),
).withColumn("has_valid_email", F.col("email").contains("@"))

customer_key_audit = (
    customers_typed.where(F.col("customer_id").isNull() | (F.trim("customer_id") == ""))
    .withColumn("audit_reason", F.lit("Missing customer_id"))
)
valid_customers = customers_typed.where(F.col("customer_id").isNotNull() & (F.trim("customer_id") != ""))
email_audit = valid_customers.where(~F.col("has_valid_email")).withColumn("audit_reason", F.lit("Email does not contain @"))
overwrite_delta(customer_key_audit, table_name(SILVER_SCHEMA, "customer_key_audit"))
overwrite_delta(email_audit, table_name(SILVER_SCHEMA, "customer_email_audit"))

customers_destination = table_name(SILVER_SCHEMA, "customers")
if spark.catalog.tableExists(customers_destination):
    (
        DeltaTable.forName(spark, customers_destination)
        .alias("target")
        .merge(valid_customers.alias("source"), "target.customer_id = source.customer_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    overwrite_delta(valid_customers, customers_destination)

silver_customers = spark.table(customers_destination)
show_count("Silver customers", silver_customers)
show_count("Customer key audit", customer_key_audit)
show_count("Customer email audit", email_audit)

Silver inventory: 4,848 rows
Inventory records excluded because stock quantity is null: 152
Inventory records retained with negative stock warning: 0
Silver customers: 10,000 rows
Customer key audit: 0 rows
Customer email audit: 0 rows


0

## Lets create the Gold daily revenue table
### Why is this used?
Gold tables contain business metrics, not raw operational rows. `daily_revenue` lets a report answer which regions and product categories are contributing the most sales on each day.

In [0]:
silver_orders = spark.table(table_name(SILVER_SCHEMA, "orders"))
silver_order_items = spark.table(table_name(SILVER_SCHEMA, "order_items"))
silver_inventory = spark.table(table_name(SILVER_SCHEMA, "inventory"))

if silver_orders.limit(1).count() == 0:
    raise ValueError(
        "No valid orders reached Silver. Review ecommerce_silver.orders_quarantine before creating Gold metrics."
    )

daily_revenue = (
    silver_order_items.alias("item")
    .join(silver_orders.alias("order"), F.col("item.order_id") == F.col("order.order_id"), "inner")
    .where(F.col("order.status") != "cancelled")
    .groupBy(F.to_date("order.order_date").alias("order_date"), F.col("order.region").alias("region"), F.col("item.category").alias("category"))
    .agg(
        F.round(F.sum("item.line_total"), 2).alias("total_revenue"),
        F.countDistinct("order.order_id").alias("order_count"),
        F.round(F.sum("item.line_total") / F.countDistinct("order.order_id"), 2).alias("average_order_value"),
    )
)
overwrite_delta(daily_revenue, table_name(GOLD_SCHEMA, "daily_revenue"))
show_count("Gold daily revenue", daily_revenue)
display(daily_revenue.orderBy(F.desc("total_revenue")).limit(10))

Gold daily revenue: 4,240 rows


order_date,region,category,total_revenue,order_count,average_order_value
2025-02-22,North,Grocery,2697689.01,41,65797.29
2025-03-24,West,Toys,2588251.44,38,68111.88
2025-04-05,Central,Toys,2486838.36,24,103618.26
2025-01-09,West,Sports,2481589.84,37,67070.0
2025-02-27,South,Clothing,2453279.47,37,66304.85
2025-04-15,East,Sports,2444204.82,36,67894.58
2025-02-28,East,Home & Kitchen,2410981.88,37,65161.67
2025-03-20,South,Grocery,2387378.75,36,66316.08
2025-01-02,East,Toys,2369454.26,36,65818.17
2025-04-05,North,Electronics,2337574.59,35,66787.85


## Lets create the Gold fulfillment KPI table
This table makes delivery, cancellation, and shipment rates comparable across warehouses and regions.

In [0]:
fulfillment_base = silver_orders.groupBy(
    F.to_date("order_date").alias("order_date"), "warehouse_id", "region"
).agg(
    F.count("order_id").alias("total_orders"),
    F.sum(F.when(F.col("status") == "delivered", 1).otherwise(0)).alias("delivered_orders"),
    F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled_orders"),
    F.sum(F.when(F.col("status") == "shipped", 1).otherwise(0)).alias("shipped_orders"),
)
fulfillment_kpi = fulfillment_base.withColumn(
    "delivery_rate_pct", F.round(100 * F.col("delivered_orders") / F.col("total_orders"), 2)
).withColumn(
    "cancellation_rate_pct", F.round(100 * F.col("cancelled_orders") / F.col("total_orders"), 2)
).withColumn(
    "shipment_rate_pct", F.round(100 * F.col("shipped_orders") / F.col("total_orders"), 2)
)
overwrite_delta(fulfillment_kpi, table_name(GOLD_SCHEMA, "fulfillment_kpi"))
show_count("Gold fulfillment KPI", fulfillment_kpi)
display(fulfillment_kpi.orderBy("order_date", "warehouse_id").limit(10))

Gold fulfillment KPI: 2,650 rows


order_date,warehouse_id,region,total_orders,delivered_orders,cancelled_orders,shipped_orders,delivery_rate_pct,cancellation_rate_pct,shipment_rate_pct
2025-01-01,WH-BLR,West,12,0,2,5,0.0,16.67,41.67
2025-01-01,WH-BLR,South,13,2,3,3,15.38,23.08,23.08
2025-01-01,WH-BLR,Central,15,3,6,4,20.0,40.0,26.67
2025-01-01,WH-BLR,East,15,6,2,4,40.0,13.33,26.67
2025-01-01,WH-BLR,North,25,8,7,5,32.0,28.0,20.0
2025-01-01,WH-CHE,Central,21,5,6,3,23.81,28.57,14.29
2025-01-01,WH-CHE,North,13,5,4,1,38.46,30.77,7.69
2025-01-01,WH-CHE,South,12,3,3,3,25.0,25.0,25.0
2025-01-01,WH-CHE,West,16,5,5,1,31.25,31.25,6.25
2025-01-01,WH-CHE,East,13,7,2,2,53.85,15.38,15.38


## Lets create the Gold inventory health table
The inventory table combines current stock with demand from the last 30 days of available orders. Using the latest order date makes the result stable even when the project data is historical.

In [0]:
latest_order_date = silver_orders.select(F.max(F.to_date("order_date")).alias("latest_order_date")).first()["latest_order_date"]
recent_demand = (
    silver_order_items.alias("item")
    .join(silver_orders.alias("order"), F.col("item.order_id") == F.col("order.order_id"), "inner")
    .where((F.col("order.status") != "cancelled") & (F.to_date("order.order_date") >= F.date_sub(F.lit(latest_order_date), 30)))
    .groupBy(F.col("item.sku_id").alias("sku_id"))
    .agg(F.sum("item.quantity").alias("demand_last_30_days"))
)
inventory_health = (
    silver_inventory.join(recent_demand, "sku_id", "left")
    .fillna({"demand_last_30_days": 0})
    .withColumn(
        "stock_status",
        F.when(F.col("stock_quantity") == 0, "stockout")
        .when(F.col("stock_quantity") <= F.col("reorder_level"), "below_reorder")
        .when(F.col("stock_quantity") >= (F.col("reorder_level") * 3), "overstock")
        .otherwise("healthy"),
    )
    .withColumn("reorder_flag", F.col("stock_quantity") <= F.col("reorder_level"))
    .withColumn("inventory_value", F.round(F.col("stock_quantity") * F.col("unit_cost"), 2))
)
overwrite_delta(inventory_health, table_name(GOLD_SCHEMA, "inventory_health"))
show_count("Gold inventory health", inventory_health)
display(inventory_health.groupBy("stock_status").count().orderBy("stock_status"))

Gold inventory health: 4,848 rows


stock_status,count
below_reorder,144
healthy,269
overstock,4433
stockout,2


## Lets create the Gold customer lifetime value table
Customer LTV combines clean customer records with non-cancelled orders. It shows total spend, order frequency, days since the last order, and a transparent value segment.

In [0]:
customer_order_metrics = (
    silver_orders.where(F.col("status") != "cancelled")
    .groupBy("customer_id")
    .agg(
        F.round(F.sum("total_amount"), 2).alias("lifetime_spend"),
        F.countDistinct("order_id").alias("order_frequency"),
        F.max(F.to_date("order_date")).alias("last_order_date"),
    )
)
customer_ltv = (
    silver_customers.join(customer_order_metrics, "customer_id", "left")
    .fillna({"lifetime_spend": 0.0, "order_frequency": 0})
    .withColumn("recency_days", F.when(F.col("last_order_date").isNull(), None).otherwise(F.datediff(F.lit(latest_order_date), F.col("last_order_date"))))
    .withColumn(
        "customer_segment",
        F.when(F.col("lifetime_spend") >= 5000, "VIP")
        .when(F.col("lifetime_spend") >= 2500, "High Value")
        .when(F.col("lifetime_spend") >= 1000, "Mid Value")
        .otherwise("Low Value"),
    )
)
overwrite_delta(customer_ltv, table_name(GOLD_SCHEMA, "customer_ltv"))
show_count("Gold customer LTV", customer_ltv)
display(customer_ltv.groupBy("customer_segment").count().orderBy("customer_segment"))

Gold customer LTV: 10,000 rows


customer_segment,count
High Value,76
Low Value,464
Mid Value,50
VIP,9410


## Lets create the reconciliation tables
### Why is this used?
Reconciliation proves what entered and left each layer. The first table records table row counts. The second table shows whether orders and order items passed data-quality validation or went to quarantine.

In [0]:
reconciliation_targets = {
    "landing.orders": spark.table(table_name(LANDING_SCHEMA, "orders")),
    "landing.order_items": spark.table(table_name(LANDING_SCHEMA, "order_items")),
    "landing.customers": spark.table(table_name(LANDING_SCHEMA, "customers")),
    "landing.inventory": spark.table(table_name(LANDING_SCHEMA, "inventory")),
    "bronze.orders": spark.table(table_name(BRONZE_SCHEMA, "orders")),
    "bronze.order_items": spark.table(table_name(BRONZE_SCHEMA, "order_items")),
    "bronze.customers": spark.table(table_name(BRONZE_SCHEMA, "customers")),
    "bronze.inventory": spark.table(table_name(BRONZE_SCHEMA, "inventory")),
    "silver.orders": valid_orders,
    "silver.order_items": valid_order_items,
    "silver.customers": silver_customers,
    "silver.inventory": valid_inventory,
    "silver.orders_quarantine": invalid_orders,
    "silver.order_items_quarantine": invalid_order_items,
    "gold.daily_revenue": daily_revenue,
    "gold.fulfillment_kpi": fulfillment_kpi,
    "gold.inventory_health": inventory_health,
    "gold.customer_ltv": customer_ltv,
}

row_count_schema = StructType(
    [
        StructField("table_name", StringType(), False),
        StructField("row_count", IntegerType(), False),
        StructField("captured_at", StringType(), False),
    ]
)
row_count_rows = [(name, dataframe.count(), RUN_TIMESTAMP.isoformat()) for name, dataframe in reconciliation_targets.items()]
reconciliation_row_counts = spark.createDataFrame(row_count_rows, row_count_schema).withColumn("captured_at", F.to_timestamp("captured_at"))
overwrite_delta(reconciliation_row_counts, table_name(GOLD_SCHEMA, "reconciliation_row_counts"))

dq_summary_schema = StructType(
    [
        StructField("dataset", StringType(), False),
        StructField("bronze_row_count", IntegerType(), False),
        StructField("deduplicated_row_count", IntegerType(), False),
        StructField("duplicate_rows_excluded", IntegerType(), False),
        StructField("silver_row_count", IntegerType(), False),
        StructField("quarantined_rows", IntegerType(), False),
    ]
)
dq_summary_rows = [
    (
        "orders",
        bronze_tables["orders"].count(),
        orders_latest.count(),
        bronze_tables["orders"].count() - orders_latest.count(),
        valid_orders.count(),
        invalid_orders.count(),
    ),
    (
        "order_items",
        bronze_tables["order_items"].count(),
        order_items_latest.count(),
        bronze_tables["order_items"].count() - order_items_latest.count(),
        valid_order_items.count(),
        invalid_order_items.count(),
    ),
]
reconciliation_dq_summary = (
    spark.createDataFrame(dq_summary_rows, dq_summary_schema)
    .withColumn("pass_rate_pct", F.round(100 * F.col("silver_row_count") / F.col("deduplicated_row_count"), 2))
    .withColumn("quarantine_rate_pct", F.round(100 * F.col("quarantined_rows") / F.col("deduplicated_row_count"), 2))
    .withColumn("captured_at", F.lit(RUN_TIMESTAMP).cast("timestamp"))
)
overwrite_delta(reconciliation_dq_summary, table_name(GOLD_SCHEMA, "reconciliation_dq_summary"))

display(reconciliation_row_counts.orderBy("table_name"))
display(reconciliation_dq_summary.orderBy("dataset"))

table_name,row_count,captured_at
bronze.customers,10000,2026-08-13T16:02:13.952Z
bronze.inventory,5000,2026-08-13T16:02:13.952Z
bronze.order_items,200000,2026-08-13T16:02:13.952Z
bronze.orders,50150,2026-08-13T16:02:13.952Z
gold.customer_ltv,10000,2026-08-13T16:02:13.952Z
gold.daily_revenue,4240,2026-08-13T16:02:13.952Z
gold.fulfillment_kpi,2650,2026-08-13T16:02:13.952Z
gold.inventory_health,4848,2026-08-13T16:02:13.952Z
landing.customers,10000,2026-08-13T16:02:13.952Z
landing.inventory,5000,2026-08-13T16:02:13.952Z


dataset,bronze_row_count,deduplicated_row_count,duplicate_rows_excluded,silver_row_count,quarantined_rows,pass_rate_pct,quarantine_rate_pct,captured_at
order_items,200000,200000,0,195044,4956,97.52,2.48,2026-08-13T16:02:13.952Z
orders,50150,50000,150,40591,9409,81.18,18.82,2026-08-13T16:02:13.952Z


## Lets review the final project output
The Gold schema now contains six business and audit tables:

`daily_revenue`
`fulfillment_kpi`
`inventory_health`
`customer_ltv`
`reconciliation_row_counts`
`reconciliation_dq_summary`

Open the `ecommerce_gold` schema in Catalog Explorer to preview them. You can use these tables directly in Databricks SQL dashboards or Power BI.

In [0]:
print("Pipeline completed.")
print(f"Explore Gold tables in: {CATALOG_NAME}.{GOLD_SCHEMA}")

Pipeline completed.
Explore Gold tables in: testad.ecommerce_gold
